# AxonScope Colab GPU Hotpaths

Use this notebook in a Google Colab GPU runtime. It clones the moving `bench-colab` branch, installs AxonScope, runs the warm hotpath scale probe, stores results under `benchmark/results/hotpaths/` inside the Colab checkout, zips the run folder, and downloads it directly through the browser.

Local setup before running this notebook:

```bash
git add -A
git commit -m "Benchmark Colab run"
make bench-colab-push
```

After download, unzip the archive into your local `benchmark/results/hotpaths/` folder.

In [ ]:
import datetime
import pathlib
import shutil
import subprocess

from google.colab import files

# Replace this once with the real repository URL.
REPO_URL = "https://github.com/YOUR_USER/YOUR_REPO.git"
BRANCH = "bench-colab"
PKG_DIR = pathlib.Path("/content/AxonScope")

WORKLOAD = "all"
PRESET = "scale"
WARMUPS = 1

run_id = datetime.datetime.now().strftime("colab_gpu_%Y%m%d_%H%M%S")


def sh(command, cwd=None):
    print(f"\n$ {command}")
    subprocess.run(command, shell=True, cwd=cwd, check=True)


sh(f"rm -rf {PKG_DIR}")
sh(f"git clone --depth 1 --branch {BRANCH} {REPO_URL} {PKG_DIR}")
sh("git rev-parse --short HEAD", cwd=PKG_DIR)

sh("python -m pip install -U pip", cwd=PKG_DIR)
sh('python -m pip install -e ".[examples,benchmark]"', cwd=PKG_DIR)
sh("nvidia-smi || true", cwd=PKG_DIR)
sh(
    "python - <<'PY'\n"
    "import jax\n"
    "print('jax backend:', jax.default_backend())\n"
    "print('jax devices:', jax.devices())\n"
    "if jax.default_backend() != 'gpu':\n"
    "    raise SystemExit('Colab runtime is not using a GPU backend.')\n"
    "PY",
    cwd=PKG_DIR,
)

results_root = PKG_DIR / "benchmark/results/hotpaths"
results_root.mkdir(parents=True, exist_ok=True)
sh(
    "python benchmark/hotpaths/run.py "
    f"--workload {WORKLOAD} "
    f"--preset {PRESET} "
    f"--warmups {WARMUPS} "
    f"--prefix {run_id} "
    "--out-dir benchmark/results/hotpaths "
    "--no-print-summary",
    cwd=PKG_DIR,
)

run_dir = results_root / run_id
zip_path = shutil.make_archive(
    str(results_root / run_id),
    "zip",
    root_dir=results_root,
    base_dir=run_id,
)
print(f"\nResults folder: {run_dir}")
print(f"Archive: {zip_path}")
files.download(zip_path)